In [1]:
import gzip
import json
import dateutil.parser
import random
import numpy as np
from collections import defaultdict

In [2]:
import homework2

In [3]:
root = "/home/traxphil/Workspace/UCSD/recommender_systems/rs_project_space"

In [4]:
def parseData(fname):
    for l in open(fname):
        yield eval(l)

In [5]:
data = list(parseData(root + "/datasets/beer_50000.json"))
len(data)
data[0]

{'review/appearance': 2.5,
 'beer/style': 'Hefeweizen',
 'review/palate': 1.5,
 'review/taste': 1.5,
 'beer/name': 'Sausa Weizen',
 'review/timeUnix': 1234817823,
 'beer/ABV': 5.0,
 'beer/beerId': '47986',
 'beer/brewerId': '10325',
 'review/timeStruct': {'isdst': 0,
  'mday': 16,
  'hour': 20,
  'min': 57,
  'sec': 3,
  'mon': 2,
  'year': 2009,
  'yday': 47,
  'wday': 0},
 'review/overall': 1.5,
 'review/text': 'A lot of foam. But a lot.\tIn the smell some banana, and then lactic and tart. Not a good start.\tQuite dark orange in color, with a lively carbonation (now visible, under the foam).\tAgain tending to lactic sourness.\tSame for the taste. With some yeast and banana.',
 'user/profileName': 'stcules',
 'review/aroma': 2.0}

In [6]:
random.seed(0)
random.shuffle(data)

In [7]:
dataTrain = data[:25000]
dataValid = data[25000:37500]
dataTest = data[37500:]

In [8]:
categoryCounts = defaultdict(int)
for d in data:
    categoryCounts[d['beer/style']] += 1

In [ ]:
categories = [c for c in categoryCounts if categoryCounts[c] > 1000]

In [10]:
catID = dict(zip(list(categories),range(len(categories))))

In [11]:
catID

{'American Amber / Red Ale': 0,
 'American Porter': 1,
 'Euro Pale Lager': 2,
 'Fruit / Vegetable Beer': 3,
 'English Pale Ale': 4,
 'Rauchbier': 5,
 'American Pale Ale (APA)': 6,
 'Hefeweizen': 7,
 'Baltic Porter': 8,
 'Scotch Ale / Wee Heavy': 9,
 'American IPA': 10,
 'Old Ale': 11,
 'American Double / Imperial IPA': 12,
 'American Double / Imperial Stout': 13,
 'Euro Strong Lager': 14,
 'Czech Pilsener': 15,
 'Rye Beer': 16,
 'Russian Imperial Stout': 17,
 'American Blonde Ale': 18,
 'Pumpkin Ale': 19,
 'Doppelbock': 20,
 'Extra Special / Strong Bitter (ESB)': 21,
 'Munich Helles Lager': 22,
 'American Barleywine': 23,
 'Maibock / Helles Bock': 24,
 'MÃ¤rzen / Oktoberfest': 25,
 'Belgian Strong Pale Ale': 26,
 'American Stout': 27,
 'Wheatwine': 28,
 'English Porter': 29,
 'German Pilsener': 30,
 'English Bitter': 31,
 'American Adjunct Lager': 32,
 'English Brown Ale': 33,
 'Tripel': 34,
 'Winter Warmer': 35,
 'Light Lager': 36,
 'American Brown Ale': 37}

In [12]:
def testQ1():
    mod, validBER, testBER = homework2.Q1(catID, dataTrain, dataValid, dataTest)
    return validBER, testBER

In [13]:
testQ1()

(np.float64(0.09954919086582126), np.float64(0.09748007766145239))

In [14]:
def testQ2():
    mod, validBER, testBER = homework2.Q2(catID, dataTrain, dataValid, dataTest)
    return validBER, testBER

In [15]:
testQ2()

(np.float64(0.0960286785563366), np.float64(0.0971724666212078))

In [16]:
def testQ3():
    mod, validBER, testBER = homework2.Q3(catID, dataTrain, dataValid, dataTest)
    return validBER, testBER

In [17]:
testQ3()

(np.float64(0.0960286785563366), np.float64(0.0971724666212078))

In [18]:
def testQ4():
    testBER_noCat, testBER_noReview, testBER_noLength = homework2.Q4(1, catID, dataTrain, dataValid, dataTest)
    return testBER_noCat, testBER_noReview, testBER_noLength

In [ ]:
testQ4()

In [20]:
path = root + "/amazon_reviews_us_Musical_Instruments_v1_00.tsv.gz"
f = gzip.open(path, 'rt', encoding="utf8")

header = f.readline()
header = header.strip().split('\t')

FileNotFoundError: [Errno 2] No such file or directory: '/home/traxphil/Workspace/UCSD/recommender_systems/rs_project_space/amazon_reviews_us_Musical_Instruments_v1_00.tsv.gz'

In [ ]:
header

In [ ]:
review_dataset = []

pairsSeen = set()

for line in f:
    fields = line.strip().split('\t')
    d = dict(zip(header, fields))
    ui = (d['customer_id'], d['product_id'])
    if ui in pairsSeen:
        print("Skipping duplicate user/item:", ui)
        continue
    pairsSeen.add(ui)
    d['star_rating'] = int(d['star_rating'])
    d['helpful_votes'] = int(d['helpful_votes'])
    d['total_votes'] = int(d['total_votes'])
    review_dataset.append(d)

In [ ]:
reviewDataTrain = review_dataset[:int(len(review_dataset)*0.9)]
reviewDataTest = review_dataset[int(len(review_dataset)*0.9):]

In [ ]:
usersPerItem = defaultdict(set) # Maps an item to the users who rated it
itemsPerUser = defaultdict(set) # Maps a user to the items that they rated
itemNames = {}
ratingDict = {} # To retrieve a rating for a specific user/item pair
reviewsPerUser = defaultdict(list)

for d in reviewDataTrain:
    user,item = d['customer_id'], d['product_id']
    usersPerItem[item].add(user)
    itemsPerUser[user].add(item)
    reviewsPerUser[user].append(d)

for d in review_dataset:
    user,item = d['customer_id'], d['product_id']
    ratingDict[(user,item)] = d['star_rating']
    itemNames[item] = d['product_title']

In [ ]:
def testQ5():
    # Note: your solution is autograded based on the relevance scores, not the items themselves
    return homework2.mostSimilar("B00KCHRKD6", 10, usersPerItem)

In [ ]:
#testQ5()

In [ ]:
ratingMean = homework2.getMeanRating(reviewDataTrain)

userAverages = homework2.getUserAverages(itemsPerUser, ratingDict)

itemAverages = homework2.getItemAverages(usersPerItem, ratingDict)

In [ ]:
def testQ6():
    alwaysPredictMean = [ratingMean for d in reviewDataTest]
    
    simPredictions = [homework2.predictRating(d['customer_id'],
                                              d['product_id'],
                                              ratingMean,
                                              reviewsPerUser,
                                              usersPerItem,
                                              itemsPerUser,
                                              userAverages,
                                              itemAverages) for d in reviewDataTest]

    labels = [d['star_rating'] for d in reviewDataTest]
    
    # Autograder checks the MSE of your predictions and (some of) the simPrediction values

In [ ]:
#testQ6()

In [ ]:
def testQ7():
    alwaysPredictMean = [ratingMean for d in reviewDataTest]
    
    simPredictions = [homework2.predictRating(d['customer_id'],
                                              d['product_id'],
                                              ratingMean,
                                              reviewsPerUser,
                                              usersPerItem,
                                              itemsPerUser,
                                              userAverages,
                                              itemAverages) for d in reviewDataTest]
    

    q7Predictions = [homework2.predictRatingQ7(d['customer_id'],
                                               d['product_id'],
                                               ratingMean,
                                               reviewsPerUser,
                                               usersPerItem,
                                               itemsPerUser,
                                               userAverages,
                                               itemAverages) for d in reviewDataTest]
    
    labels = [d['star_rating'] for d in reviewDataTest]
    
    m1 = homework2.MSE(simPredictions, labels)
    m2 = homework2.MSE(q7Predictions, labels)
    m3 = homework2.MSE(alwaysPredictMean, labels)
    
    # Autograder checks whether your solution is better than either the Q6 or a naive solution
    return 1.0 * ((m2 < m1) and (m2 < m3))